[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/protosome/convergent_overlaps_aa_change/blob/main/convergent_overlapping_gene_generation.ipynb)


In [ ]:
###############################################
# Download and install supporting tools
###############################################
!git clone https://github.com/protosome/convergent_overlaps_aa_change.git
%cd convergent_overlaps_aa_change

from paths import ROOT_DIR
print("Root dir is:", ROOT_DIR)

!pip install biopython
!pip install fair-esm

%cd {ROOT_DIR}/s4pred
!wget http://bioinfadmin.cs.ucl.ac.uk/downloads/s4pred/weights.tar.gz
!tar -xvzf weights.tar.gz

%cd {ROOT_DIR}

In [ ]:
#@title ###**Load dependencies**.

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
import torch.nn as nn
import tensorflow as tf
import numpy as np
import pandas as pd
import json
from Bio import pairwise2
from Bio.Seq import Seq
from Bio.Seq import CodonTable
import itertools as it
import pandas as pd
import math
import random
import pickle
from protsub_matrix import prot_sub_matrix, calculate_protsub_similarity
from blosum62_matrix import blosum62_matrix, calculate_blosum62_similarity
from running_s4pred import predict_secondary_structure # This loads the S4PRED function to run as a subprocess, outputting only the structure prediction sequence
from running_s4pred_batch_fast import predict_secondary_structure_with_progress # Updated version of the S4PRED function, in faster batches
from transformer_encoder_model import TransformerModel, SinusoidalPositionalEncoding, TransformerBlock
import openpyxl
import time, math
import torch.nn.functional as F
from typing import Optional, List, Tuple, Dict, Any, Iterable
from skimage.metrics import structural_similarity as ssim
import re
import io
import hashlib
import sys
from datetime import datetime
from contextlib import redirect_stdout


In [ ]:
#@title ###**Main and supporting functions**.

from paths import ROOT_DIR
from overlap_runtime.notebook_api import (
    create_runtime,
    load_pairs_table,
    optimize_pair_and_save,
    run_batch_from_table,
)

# Build notebook runtime once so torch unpickle class registration and runtime state are available.
runtime = create_runtime(
    ROOT_DIR,
    esm_device="auto",
    esm_batch_size=4,
    esm_autocast=False,
    require_esm=True,
    reset_caches_per_row=False,
    debug_checkpoints=False,
)

print("[INFO] Modular runtime initialized.")
print("[INFO] Notebook now acts as a wrapper over overlap_runtime modules.")


In [ ]:
#@title ### Run Overlap
#@markdown Provide parameters below and run the cell.
working_dir = "test_results" #@param {type:"string"}
excel_path = "test_results/aa_1_aa_2.xlsx" #@param {type:"string"}
upload_excel = False #@param {type:"boolean"}

#@markdown Overlap nucleotide length selection (from 199 to 312 available)
overlap_length_selected = "310" #@param {type:"string"}

#@markdown Row slicing (Excel-style; 1-based)
start_row = 1 #@param {type:"integer"}
end_row = 1 #@param {type:"integer"} # 0 means "to end"

#@markdown Multi-objective optimization weight controls
use_row_weights = True #@param {type:"boolean"}
ss_w = 0.15 #@param {type:"number"}
sub_w = 0.15 #@param {type:"number"}
aln_w = 0.10 #@param {type:"number"}
esm_w = 0.60 #@param {type:"number"}
normalize_override_weights = False #@param {type:"boolean"}

#@markdown Pass iteration controls (optimally, leave first pass at 1)
FIRST_PASS_ITERS  = 1   #@param {type:"integer"}
SECOND_PASS_ITERS = 75  #@param {type:"integer"}

#@markdown Misc
reset_caches_per_row = False #@param {type:"boolean"}

# ---- end of form fields ----

import os
from google.colab import files
from paths import ROOT_DIR
from overlap_runtime.notebook_api import create_runtime, run_batch_from_table

if upload_excel:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    fname = list(uploaded.keys())[0]
    local_working_dir = os.path.join(str(ROOT_DIR), working_dir)
    os.makedirs(local_working_dir, exist_ok=True)
    target_file = os.path.join(local_working_dir, fname)
    with open(target_file, "wb") as fh:
        fh.write(uploaded[fname])
else:
    target_file = excel_path
    if not os.path.isabs(target_file):
        target_file = os.path.join(str(ROOT_DIR), target_file)

runtime = create_runtime(
    ROOT_DIR,
    esm_device="auto",
    esm_batch_size=4,
    esm_autocast=False,
    require_esm=True,
    reset_caches_per_row=bool(reset_caches_per_row),
    debug_checkpoints=False,
)

outputs = run_batch_from_table(
    excel_path=target_file,
    working_dir=os.path.join(str(ROOT_DIR), working_dir),
    overlap_length_selected=overlap_length_selected,
    start_row=int(start_row),
    end_row=int(end_row),
    use_row_weights=bool(use_row_weights),
    ss_w=float(ss_w),
    sub_w=float(sub_w),
    aln_w=float(aln_w),
    esm_w=float(esm_w),
    normalize_override_weights=bool(normalize_override_weights),
    first_pass_iters=int(FIRST_PASS_ITERS),
    second_pass_iters=int(SECOND_PASS_ITERS),
    archive=True,
)

print(f"[FINISHED] Run complete. Generated files: {len(outputs)}")
